# Dijkstra con cubetas de Dial — demostraciónTarea individual: hallar la distancia entre dos vértices leyendo el grafo de un archivo de texto.Límites del enunciado: `1 ≤ V ≤ 1.000.000`, `1 ≤ E ≤ 500.000`, pesos enteros `1 ≤ w ≤ 100`.**Implementación:** C, algoritmo de **Dial** en lugar de un heap. Como los pesos son enteros de 1 a 100,todas las distancias que hay en la cola en un instante caben en la ventana `[d, d+100]`, así que**101 cubetas circulares** bastan y sacar el mínimo cuesta *O(1)* en vez de *O(log n)*, sin comparaciones.Otras decisiones:- **máscara de 128 bits + `__builtin_ctzll`** para saltar de golpe a la siguiente cubeta no vacía;- **borrado perezoso** en vez de `decrease-key`: se insertan duplicados y se descartan con `dist[u] != dist_actual`;- **terminación temprana** al extraer el destino;- grafo en **arreglos planos** (`head`/`next`, `tw = (destino<<7)|peso`), construido en la misma pasada del parseo;- entrada con **`mmap`** y conversión de dígitos a mano (sin `scanf`), porque con 1.000.000 de etiquetas  el archivo pesa ~15 MB y **leerlo cuesta más que ejecutar Dijkstra**.Repositorio: https://github.com/JLtz00/dijkstra-dial

## 1. Entorno y compilaciónSe muestra en qué máquina de Colab se midió, porque los tiempos dependen de ella.

In [ ]:
!git clone -q https://github.com/JLtz00/dijkstra-dial.git%cd dijkstra-dial!gcc -O2 -march=native -o dijkstra dijkstra.c && echo "compilado OK"!lscpu | grep -E "^Model name|^CPU\(s\):" ; free -h | head -2

## 2. El ejemplo del enunciado`V=10`, `E=2`, camino de `3` a `7`: las aristas `3-5` (14) y `5-7` (7) dan **21**.

In [ ]:
%%writefile ejemplo.txt10 23 7123456789103 5 145 7 7

In [ ]:
!./dijkstra ejemplo.txt

## 3. Correctitud`test.sh` comprueba:- los casos borde: `s == t`, grafo sin camino, una sola arista, lectura por `stdin`;- que las **cuatro** colas de prioridad implementadas den el mismo resultado;- **40 grafos aleatorios** contra un Dijkstra de referencia escrito en Python (`heapq`);- un grafo de `V = 1.000.000` con todos los pesos en 100, cuya distancia es exactamente  5×10⁷ (verifica que no haya desbordamiento en 32 bits).

In [ ]:
!./test.sh

## 4. Tiempo con la entrada más grande del enunciado`V = 1.000.000`, `E = 500.000`: un archivo de ~15 MB con un millón de etiquetas.`TIEMPO=1` hace que el programa imprima en **stderr** el desglose de tiempos(cuesta 3 llamadas a `clock_gettime`, unos 93 ns, es decir 0,0001 ms);por **stdout** sale únicamente la distancia.

In [ ]:
# cadena de 500.000 aristas de peso 100 -> distancia 50.000.000import timeV, E = 1_000_000, 500_000t0 = time.time()with open("grande.txt", "w") as f:    f.write(f"{V} {E}\n1 {E+1}\n")    f.write("".join(f"n{i}\n" for i in range(1, V + 1)))    f.write("".join(f"{i} {i+1} 100\n" for i in range(1, E + 1)))print(f"archivo generado en {time.time()-t0:.1f} s")!ls -lh grande.txt

In [ ]:
# 5 corridas: la primera suele salir mas lenta (cache frio y CPU compartida de Colab)!for i in 1 2 3 4 5; do TIEMPO=1 ./dijkstra grande.txt; done

### Lectura de esos números`parseo + algoritmo = total`. El parseo recorre los ~15 millones de bytes del archivo uno por uno;Dijkstra solo recorre la estructura ya construida. Por eso pesan parecido: **el cuello de botellareal es la entrada, no el algoritmo**.El `total` no incluye el arranque ni el cierre del proceso (~4 ms), así que es menor que lo quemediría `time` desde fuera.

## 5. ¿Por qué Dial y no un heap?`queues.h` tiene **cuatro** colas de prioridad intercambiables sobre el mismo Dijkstra, y`bench.c` las cronometra sobre el mismo grafo ya parseado (el parseo se mide aparte):| cola | qué es ||---|---|| `dial` | 101 cubetas circulares + máscara de bits para saltar las vacías || `barrido` | las mismas cubetas, pero buscando la siguiente recorriéndolas una por una || `heap` | heap binario, clave `(dist<<20)\|nodo` empaquetada en un `uint64` || `heap4` | heap 4-ario (menos niveles y hijos contiguos: menos fallos de caché) |Se prueban seis formas de grafo (cadena larga, malla 500×500, disperso de 1M nodos y denso entres tamaños). Tarda uno o dos minutos: genera ~70 MB de grafos de prueba.

In [ ]:
!./bench.sh 5

In [ ]:
import pandas as pddf = pd.read_csv("resultados.csv")tabla = df.pivot(index="grafo", columns="variante", values="ms_med")tabla[["parseo", "Dial+mascara", "Dial+barrido", "Heap binario", "Heap 4-ario"]]

In [ ]:
from IPython.display import HTMLHTML(open("comparacion.html").read())   # el mismo cuadro como grafico

### Conclusiones del benchmark- **La máscara de bits solo importa en la cadena** (28,9 → 8,0 ms): ahí cada arista pesa ~50, así que  entre dos extracciones hay que saltar ~50 cubetas vacías; 500.000 × 50 = 25 millones de pasos en  vacío. En la malla y en el denso la frontera tiene miles de nodos con distancias contiguas y la  cubeta siguiente está casi siempre a un paso.- **El heap gana en la cadena** (2,7 vs 8,0 ms) porque ahí la cola tiene 1 o 2 elementos: `log n ≈ 1`.  **Dial gana 3x en la malla** (7,1 vs 23,2 ms), donde la frontera tiene ~1.000 nodos y cada operación  del heap salta a `i/2, i/4, …` provocando un fallo de caché por nivel.- **El heap 4-ario le gana al binario** (15,0 vs 23,2 ms) haciendo *más* comparaciones: tiene la mitad  de niveles y los 4 hijos son contiguos. En hardware moderno gana el que toca menos memoria.- Se entrega **Dial** por criterio *minimax*: su peor caso medido es 8 ms contra los 23 ms del heap.  No se sabe qué grafo usará el evaluador, así que se minimiza el peor caso.- En el grafo disperso Dijkstra tarda **0,00 ms** y el archivo 10 ms: el peor caso del algoritmo y  el peor caso del programa no son el mismo grafo.

## 6. Probar con otro archivoSube un archivo con el formato del enunciado y se ejecuta sobre él.También se puede elegir la cola: `./dijkstra archivo.txt heap`.

In [ ]:
from google.colab import filessubido = files.upload()nombre = next(iter(subido))!TIEMPO=1 ./dijkstra "$nombre"